# TAHAP 3a - LAPIS 1: Training Isolation Forest

Pada tahap ini kita akan melatih **Isolation Forest**, sebuah model **unsupervised** yang belajar mengidentifikasi anomali tanpa melihat label fraud.

**Konsep Pola B:**
- Isolation Forest mempelajari "seperti apa transaksi normal" dari data keseluruhan
- Menghasilkan **anomaly score** untuk setiap transaksi (makin tinggi = makin aneh)
- Skor anomali ini akan menjadi **SATU FITUR TAMBAHAN** di Lapis 2 (model supervised)

**Mengapa Pola B efektif:**
- Kombinasi unsupervised (mendeteksi anomali) + supervised (belajar dari label) = akurasi lebih baik
- Bukan hanya mengandalkan anomali, tapi juga pola perilaku yang terlihat dari label historis

## Import & Konfigurasi

In [1]:
import os
import sys
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

# Tambahkan path agar bisa import modul dari folder ml_pipeline
current_dir = os.path.abspath('')
if os.path.basename(current_dir) == 'ml_pipeline':
    ml_path = current_dir
else:
    ml_path = os.path.join(current_dir, 'ml_pipeline')
if ml_path not in sys.path:
    sys.path.insert(0, ml_path)

from feature_engineering import load_data, engineer_features, get_feature_matrix

#   Konfigurasi Path Model  
MODEL_PATH = os.path.join(ml_path, "models", "isolation_forest.pkl")

#   Parameter Isolation Forest  
IF_PARAMS = {
    "n_estimators": 200,
    "contamination": 0.06,       # perkiraan proporsi anomali (~ proporsi fraud)
    "max_features": 1.0,
    "random_state": 42,
    "n_jobs": -1,
}

print("[OK] Import & konfigurasi selesai")

[OK] Import & konfigurasi selesai


## Load Data & Siapkan Fitur

In [2]:
print("[1/3] Memuat & menyiapkan fitur (tanpa is_fraud)...")

# Load data mentah dari database
raw_data = load_data()

# Engineer fitur perilaku
feat = engineer_features(raw_data)

# Ambil hanya kolom fitur numerik (tanpa cashier_id, tanpa is_fraud)
X = get_feature_matrix(feat)

print(f"      {X.shape[0]:,} transaksi x {X.shape[1]} fitur")
print(f"\n  Kolom fitur:")
for i, col in enumerate(X.columns, 1):
    print(f"    {i:2d}. {col}")

[1/3] Memuat & menyiapkan fitur (tanpa is_fraud)...
      2,116 transaksi x 9 fitur

  Kolom fitur:
     1. hour_of_day
     2. is_refund
     3. time_gap_seconds
     4. txn_freq_daily
     5. refund_count_daily
     6. refund_ratio_daily
     7. amount_zscore_cashier
     8. amount_rolling_mean_5
     9. amount_deviation_from_mean


## Training Isolation Forest

In [3]:
print("\n[2/3] Melatih Isolation Forest (unsupervised)...")

# Buat pipeline: scaler + Isolation Forest
# Scaler penting agar fitur dengan skala berbeda ditangani sama
pipe = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(**IF_PARAMS)),
])

# Training (tidak melihat label is_fraud)
pipe.fit(X)

print("      Training selesai")

# Analisis hasil deteksi anomali
predictions = pipe.predict(X)  # -1 = anomali, 1 = normal
n_anomaly = int((predictions == -1).sum())
pct_anomaly = (n_anomaly / len(X)) * 100

print(f"\n  Hasil deteksi anomali:")
print(f"    Ditandai anomali    : {n_anomaly:,} transaksi ({pct_anomaly:.1f}%)")
print(f"    Ditandai normal    : {len(X) - n_anomaly:,} transaksi ({100 - pct_anomaly:.1f}%)")
print(f"\n  Catatan: ini tebakan buta IF, belum dibandingkan ke kunci jawaban (is_fraud).")
print(f"           Perbandingan baru dilakukan di Lapis 2.")


[2/3] Melatih Isolation Forest (unsupervised)...
      Training selesai

  Hasil deteksi anomali:
    Ditandai anomali    : 127 transaksi (6.0%)
    Ditandai normal    : 1,989 transaksi (94.0%)

  Catatan: ini tebakan buta IF, belum dibandingkan ke kunci jawaban (is_fraud).
           Perbandingan baru dilakukan di Lapis 2.


## Simpan Model

In [4]:
print("\n[3/3] Menyimpan model...")

# Buat direktori models jika belum ada
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# Simpan model
joblib.dump(pipe, MODEL_PATH)

print(f"      Model disimpan -> {MODEL_PATH}")
print("\n[OK] Tahap 3a selesai!")
print("\nLanjutan: Jalankan Tahap 3b (02_train_supervised.ipynb) untuk melatih Lapis 2.")


[3/3] Menyimpan model...
      Model disimpan -> c:\Users\gavin\Downloads\NEW Fraudguard\ml_pipeline\models\isolation_forest.pkl

[OK] Tahap 3a selesai!

Lanjutan: Jalankan Tahap 3b (02_train_supervised.ipynb) untuk melatih Lapis 2.
